# Applied Machine Learning Workshop: Customer Support Ticket Classification

In this workshop, we will build a practical machine learning baseline for routing customer support tickets.

The model will read a ticket's subject and body, then predict the most likely support queue. This is a common first-pass ML use case: the model does not replace a support team, but it can help with triage, routing, and prioritization.

## 1. Setup

We start by importing the libraries used throughout the notebook.

- `pandas` helps us load and inspect tabular data.
- `scikit-learn` provides the machine learning workflow.
- `matplotlib` gives us a lightweight way to visualize class balance and evaluation results.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_columns", 40)

RANDOM_STATE = 42

## 2. Load the Dataset

The dataset has been copied into this project under `data/support_tickets.csv` so the notebook can run without depending on a file in Downloads.

The path logic below works whether Jupyter starts in the project root or inside the `notebooks` folder.

In [ ]:
project_root = Path.cwd()
if not (project_root / "data" / "support_tickets.csv").exists():
    project_root = project_root.parent

data_path = project_root / "data" / "support_tickets.csv"
tickets = pd.read_csv(data_path)

print(f"Rows: {tickets.shape[0]:,}")
print(f"Columns: {tickets.shape[1]:,}")
display(tickets.head())

## 3. Inspect the Data

Before training anything, we need to understand what the dataset contains.

This is one of the most important habits in applied machine learning: inspect the data before choosing the model.

In [ ]:
display(tickets.info())

summary = pd.DataFrame(
    {
        "missing_values": tickets.isna().sum(),
        "unique_values": tickets.nunique(dropna=True),
    }
)
display(summary)

In [ ]:
print("Language counts")
display(tickets["language"].value_counts())

print("Support queue counts")
display(tickets["queue"].value_counts())

print("Priority counts")
display(tickets["priority"].value_counts())

print("Ticket type counts")
display(tickets["type"].value_counts())

## 4. Choose a Prediction Target

For this workshop, we will predict the `queue` column.

That means the model is learning a routing task: given the text of a ticket, which team or queue should handle it?

We will use English-language rows only. The original dataset also contains German tickets, which is useful for a later discussion about multilingual ML systems, but using one language keeps this 3-hour workshop focused.

In [ ]:
english_tickets = tickets[tickets["language"].str.lower() == "en"].copy()

print(f"English-language tickets: {len(english_tickets):,}")
display(english_tickets[["subject", "body", "queue", "priority", "type"]].head())

## 5. Prepare the Text and Labels

Machine learning models need features and labels.

- The feature will be the ticket text, created from `subject` plus `body`.
- The label will be `queue`, the support queue we want the model to predict.

We also remove rows where the combined text is blank or the target label is missing.

In [ ]:
model_data = english_tickets.copy()

model_data["subject"] = model_data["subject"].fillna("")
model_data["body"] = model_data["body"].fillna("")
model_data["ticket_text"] = (model_data["subject"] + "\n\n" + model_data["body"]).str.strip()

model_data = model_data[(model_data["ticket_text"] != "") & model_data["queue"].notna()].copy()

X = model_data["ticket_text"]
y = model_data["queue"]

print(f"Usable rows: {len(model_data):,}")
print(f"Number of target classes: {y.nunique()}")
display(model_data[["ticket_text", "queue"]].head())

In [ ]:
queue_counts = y.value_counts()
display(queue_counts)

ax = queue_counts.sort_values().plot(kind="barh", figsize=(9, 5), color="#4C78A8")
ax.set_title("English Ticket Count by Support Queue")
ax.set_xlabel("Ticket count")
ax.set_ylabel("Support queue")
plt.tight_layout()
plt.show()

Class balance matters. If one support queue appears far more often than another, the model may learn to favor the larger classes.

We will use a weighted Logistic Regression model later so smaller classes still influence training.

## 6. Split into Training and Test Sets

We train the model on one portion of the data and evaluate it on a separate holdout set.

This helps us estimate how the model performs on tickets it did not memorize during training.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")

## 7. Build a Baseline Model

This baseline uses two steps:

1. `TfidfVectorizer` converts text into numeric features based on word importance.
2. `LogisticRegression` learns a classifier from those features.

A baseline model should be simple enough to understand, fast enough to run, and useful enough to compare against future improvements.

In [ ]:
model = Pipeline(
    steps=[
        (
            "tfidf",
            TfidfVectorizer(
                stop_words="english",
                max_features=10000,
                ngram_range=(1, 2),
                min_df=2,
            ),
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

model

## 8. Train the Model

Training means fitting the pipeline to examples where we already know the correct queue.

The vectorizer learns the vocabulary from the training text. The classifier learns patterns that connect words and phrases to support queues.

In [ ]:
model.fit(X_train, y_train)
print("Model training complete.")

## 9. Evaluate the Model

Accuracy is useful, but it is not enough by itself.

We also look at precision, recall, and F1 score for each queue. This helps us see which queues the model handles well and which ones are more difficult.

In [ ]:
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"Accuracy: {accuracy:.3f}")

report = classification_report(y_test, predictions, output_dict=True)
report_df = pd.DataFrame(report).transpose()
display(report_df)

In [ ]:
labels = list(model.classes_)
matrix = confusion_matrix(y_test, predictions, labels=labels)

fig, ax = plt.subplots(figsize=(10, 8))
image = ax.imshow(matrix, cmap="Blues")

ax.set_title("Confusion Matrix: Actual Queue vs Predicted Queue")
ax.set_xlabel("Predicted queue")
ax.set_ylabel("Actual queue")
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_yticklabels(labels)

fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

confusion_df = pd.DataFrame(matrix, index=labels, columns=labels)
display(confusion_df)

## 10. Inspect Mistakes

Aggregate metrics tell us how the model performs overall. Looking at individual mistakes tells us why.

In a real workflow, this is where you often discover unclear labels, overlapping categories, missing context, or business rules that the model cannot infer from text alone.

In [ ]:
test_results = pd.DataFrame(
    {
        "ticket_text": X_test,
        "actual_queue": y_test,
        "predicted_queue": predictions,
    }
)

probabilities = model.predict_proba(X_test)
test_results["prediction_confidence"] = probabilities.max(axis=1)

mistakes = test_results[test_results["actual_queue"] != test_results["predicted_queue"]].copy()
mistakes["text_preview"] = mistakes["ticket_text"].str.replace("\n", " ", regex=False).str.slice(0, 300)

print(f"Misclassified rows: {len(mistakes):,}")
display(
    mistakes.sort_values("prediction_confidence", ascending=False)[
        ["actual_queue", "predicted_queue", "prediction_confidence", "text_preview"]
    ].head(10)
)

High-confidence mistakes are especially useful to inspect. They can show where the model is strongly wrong, which is more concerning than a low-confidence uncertain prediction.

## 11. Try a Few Custom Examples

Now we can test a few short examples that resemble incoming tickets.

These examples are not a substitute for formal evaluation, but they make the model behavior easier to discuss.

In [ ]:
custom_examples = pd.Series(
    [
        "I was charged twice for my monthly subscription and need help with my invoice.",
        "The application is unavailable for all users and our team cannot access the portal.",
        "Can you explain whether this product integrates with our CRM and analytics tools?",
        "I need to return a device that arrived damaged and exchange it for a replacement.",
    ]
)

custom_predictions = model.predict(custom_examples)
custom_probabilities = model.predict_proba(custom_examples)
custom_confidence = custom_probabilities.max(axis=1)

custom_results = pd.DataFrame(
    {
        "ticket_text": custom_examples,
        "predicted_queue": custom_predictions,
        "confidence": custom_confidence,
    }
)

display(custom_results)

## 12. Discuss Real-World Use

A model like this is best treated as an assistant, not an autonomous decision-maker.

Useful support workflows could include:

- suggesting a queue during ticket intake
- prioritizing tickets for human review
- flagging uncertain tickets for manual routing
- helping managers understand ticket mix over time

Before using this in production, an organization would need to address:

- privacy and data governance
- human review for customer-impacting actions
- monitoring for changing support patterns
- retraining with fresh labeled examples
- clear handling of low-confidence predictions
- review of model errors by business stakeholders

## 13. Optional Extension: How This Differs From Image/Logo ML

This workshop focused on text classification. A logo normalization workflow would look different.

Text classification uses words and phrases as features. Image workflows use pixels, edges, colors, shapes, and visual patterns.

Logo normalization may involve background removal, resizing, color normalization, segmentation, vectorization, quality checks, and print-readiness rules. That makes it a good future computer vision workshop, but too much to combine deeply with this 3-hour text ML session.

## Closing Takeaways

- Machine learning is a workflow, not a single function call.
- Data inspection and preparation shape the quality of the model.
- A simple baseline is valuable because it gives you something measurable.
- Accuracy alone can hide important failures.
- Real-world use requires monitoring, governance, retraining, and human oversight.